# Notebook 1: Building & Deploying a Restaurant Assistant Agent
## Strands SDK + Amazon Bedrock Agentcore

This notebook walks through the complete process of:
1. Setting up the environment and dependencies
2. Defining agent tools (booking, retrieval, time)
3. Building the Strands SDK agent
4. Connecting to Amazon Bedrock LLMs
5. Integrating a Knowledge Base with OpenSearch and S3
6. Deploying the agent runtime on Amazon Bedrock Agentcore

**Architecture Reference:**
- Agent Framework: Strands SDK (local environment)
- LLM Backend: Amazon Bedrock
- Data Store: Amazon DynamoDB
- Knowledge Base: Amazon Bedrock Knowledge Base → OpenSearch Serverless → S3
- Observability: Arize AX (covered in Notebook 2)

---

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install strands-agents strands-agents-tools boto3 python-dotenv

In [ ]:
import os
import json
import boto3
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()

# AWS Configuration
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
DYNAMODB_TABLE_NAME = os.getenv("DYNAMODB_TABLE_NAME", "restaurant-bookings")
KNOWLEDGE_BASE_ID = os.getenv("KNOWLEDGE_BASE_ID", "your-kb-id-here")
S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME", "your-restaurant-docs-bucket")

print(f"Region: {AWS_REGION}")
print(f"DynamoDB Table: {DYNAMODB_TABLE_NAME}")
print(f"Knowledge Base ID: {KNOWLEDGE_BASE_ID}")

## 2. AWS Resource Setup

Before building the agent, ensure the following AWS resources are provisioned:
- **DynamoDB Table** for booking storage
- **S3 Bucket** with restaurant documents (menus, policies, FAQs)
- **OpenSearch Serverless** collection for vector search
- **Amazon Bedrock Knowledge Base** connected to S3 → OpenSearch

The cell below creates the DynamoDB table if it doesn't exist.

In [ ]:
# Initialize DynamoDB resource
dynamodb = boto3.resource("dynamodb", region_name=AWS_REGION)

def create_bookings_table():
    """Create the DynamoDB table for restaurant bookings if it doesn't exist."""
    try:
        table = dynamodb.create_table(
            TableName=DYNAMODB_TABLE_NAME,
            KeySchema=[
                {"AttributeName": "booking_id", "KeyType": "HASH"}
            ],
            AttributeDefinitions=[
                {"AttributeName": "booking_id", "AttributeType": "S"}
            ],
            BillingMode="PAY_PER_REQUEST"
        )
        table.wait_until_exists()
        print(f"Table '{DYNAMODB_TABLE_NAME}' created successfully.")
    except dynamodb.meta.client.exceptions.ResourceInUseException:
        print(f"Table '{DYNAMODB_TABLE_NAME}' already exists.")
    
    return dynamodb.Table(DYNAMODB_TABLE_NAME)

bookings_table = create_bookings_table()

## 3. Define Agent Tools

The restaurant assistant agent uses four tools:
- **create_booking()** — Create a new restaurant reservation
- **get_booking_details()** — Retrieve details of an existing booking
- **delete_booking()** — Cancel a reservation
- **current_time()** — Get the current date and time

Additionally, the agent uses a **retrieve()** tool connected to the Bedrock Knowledge Base for answering questions about restaurants, menus, and policies.

In [ ]:
import uuid
from strands import tool


@tool
def create_booking(
    restaurant_name: str,
    customer_name: str,
    party_size: int,
    date: str,
    time: str,
    special_requests: str = ""
) -> dict:
    """
    Create a new restaurant booking.
    
    Args:
        restaurant_name: Name of the restaurant
        customer_name: Name of the customer making the reservation
        party_size: Number of guests
        date: Reservation date (YYYY-MM-DD)
        time: Reservation time (HH:MM)
        special_requests: Any special requests or dietary requirements
    
    Returns:
        dict with booking confirmation details
    """
    booking_id = str(uuid.uuid4())[:8]
    
    item = {
        "booking_id": booking_id,
        "restaurant_name": restaurant_name,
        "customer_name": customer_name,
        "party_size": party_size,
        "date": date,
        "time": time,
        "special_requests": special_requests,
        "status": "confirmed",
        "created_at": datetime.now(timezone.utc).isoformat()
    }
    
    bookings_table.put_item(Item=item)
    
    return {
        "status": "success",
        "booking_id": booking_id,
        "message": f"Booking confirmed at {restaurant_name} for {customer_name}, "
                   f"party of {party_size} on {date} at {time}."
    }


print("Tool 'create_booking' defined.")

In [ ]:
@tool
def get_booking_details(booking_id: str) -> dict:
    """
    Retrieve details of an existing restaurant booking.
    
    Args:
        booking_id: The unique identifier of the booking
    
    Returns:
        dict with booking details or error message
    """
    response = bookings_table.get_item(Key={"booking_id": booking_id})
    
    if "Item" in response:
        return {"status": "found", "booking": response["Item"]}
    else:
        return {"status": "not_found", "message": f"No booking found with ID: {booking_id}"}


print("Tool 'get_booking_details' defined.")

In [ ]:
@tool
def delete_booking(booking_id: str) -> dict:
    """
    Cancel/delete a restaurant booking.
    
    Args:
        booking_id: The unique identifier of the booking to cancel
    
    Returns:
        dict with cancellation confirmation or error message
    """
    # First check if booking exists
    response = bookings_table.get_item(Key={"booking_id": booking_id})
    
    if "Item" not in response:
        return {"status": "not_found", "message": f"No booking found with ID: {booking_id}"}
    
    # Update status to cancelled
    bookings_table.update_item(
        Key={"booking_id": booking_id},
        UpdateExpression="SET #s = :status, cancelled_at = :cancelled_at",
        ExpressionAttributeNames={"#s": "status"},
        ExpressionAttributeValues={
            ":status": "cancelled",
            ":cancelled_at": datetime.now(timezone.utc).isoformat()
        }
    )
    
    return {
        "status": "success",
        "message": f"Booking {booking_id} has been cancelled."
    }


print("Tool 'delete_booking' defined.")

In [ ]:
@tool
def current_time() -> dict:
    """
    Get the current date and time in UTC.
    
    Returns:
        dict with current datetime information
    """
    now = datetime.now(timezone.utc)
    return {
        "current_datetime": now.isoformat(),
        "date": now.strftime("%Y-%m-%d"),
        "time": now.strftime("%H:%M:%S"),
        "day_of_week": now.strftime("%A"),
        "timezone": "UTC"
    }


print("Tool 'current_time' defined.")

## 4. Knowledge Base Retrieval Tool

This tool connects to the Amazon Bedrock Knowledge Base, which indexes restaurant documents stored in S3 through an OpenSearch Serverless vector index. It enables the agent to answer questions about restaurant menus, policies, locations, and hours.

In [ ]:
# Initialize Bedrock Agent Runtime client for Knowledge Base queries
bedrock_agent_runtime = boto3.client(
    "bedrock-agent-runtime",
    region_name=AWS_REGION
)


@tool
def retrieve(query: str) -> dict:
    """
    Retrieve information from the restaurant knowledge base.
    Search for restaurant details, menus, policies, locations, and hours.
    
    Args:
        query: The search query about restaurants
    
    Returns:
        dict with retrieved passages and source information
    """
    response = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={
            "vectorSearchConfiguration": {
                "numberOfResults": 5
            }
        }
    )
    
    results = []
    for result in response.get("retrievalResults", []):
        results.append({
            "content": result["content"]["text"],
            "score": result.get("score", 0),
            "source": result.get("location", {}).get("s3Location", {}).get("uri", "unknown")
        })
    
    return {
        "query": query,
        "num_results": len(results),
        "results": results
    }


print("Tool 'retrieve' defined (Bedrock Knowledge Base).")

## 5. Build the Strands Agent

Now we assemble the agent with all tools and configure it to use Amazon Bedrock as the LLM backend. The system prompt defines the agent's persona and behavioral constraints.

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel

# Configure Amazon Bedrock as the LLM provider
bedrock_model = BedrockModel(
    model_id="anthropic.claude-sonnet-4-20250514-v1:0",
    region_name=AWS_REGION,
    temperature=0.3,
    max_tokens=2048
)

# System prompt for the restaurant assistant
SYSTEM_PROMPT = """
You are a helpful restaurant assistant agent. You help customers with:
- Finding restaurants and their details (menus, hours, locations, policies)
- Making new reservations
- Looking up existing booking details
- Cancelling reservations
- Answering general questions about dining

Important behavioral guidelines:
1. Always use the retrieve tool to search for restaurant information before answering.
2. If a request is impossible or unrealistic (e.g., a restaurant on the moon), 
   acknowledge the constraint clearly and offer realistic alternatives.
3. Be polite, concise, and accurate.
4. Always confirm booking details with the customer before creating a reservation.
5. When providing restaurant information, cite the source if available.
6. If you don't have information about something, say so honestly rather than guessing.

Format your responses using <answer> tags for the final response to the user.
"""

# Create the agent with all tools
agent = Agent(
    model=bedrock_model,
    system_prompt=SYSTEM_PROMPT,
    tools=[
        create_booking,
        get_booking_details,
        delete_booking,
        current_time,
        retrieve
    ]
)

print("Restaurant Assistant Agent created successfully.")
print(f"Model: {bedrock_model.model_id}")
print(f"Tools: create_booking, get_booking_details, delete_booking, current_time, retrieve")

## 6. Test the Agent

Run some test queries to verify the agent works correctly across different scenarios.

In [ ]:
# Test 1: Basic restaurant query (uses retrieve tool)
print("=" * 60)
print("TEST 1: Restaurant Information Query")
print("=" * 60)

response = agent("What restaurants do you have in Seattle? What are their hours?")
print(response)

In [ ]:
# Test 2: Create a booking (uses create_booking + current_time tools)
print("=" * 60)
print("TEST 2: Create a Booking")
print("=" * 60)

response = agent(
    "I'd like to make a reservation at the Seattle restaurant for 4 people "
    "this Saturday at 7 PM. My name is Alex Johnson. We have one vegetarian."
)
print(response)

In [ ]:
# Test 3: Impossible request — graceful handling (edge case)
print("=" * 60)
print("TEST 3: Impossible Request (Graceful Handling)")
print("=" * 60)

response = agent("Ok now find me a restaurant on the moon.")
print(response)

In [ ]:
# Test 4: Retrieve and cancel a booking
print("=" * 60)
print("TEST 4: Retrieve & Cancel Booking")
print("=" * 60)

response = agent("Can you look up booking ID abc12345 and then cancel it?")
print(response)

## 7. Deploy on Amazon Bedrock Agentcore

Agentcore provides a managed runtime for deploying agents at scale. Below we package the agent and deploy it to Agentcore.

**Prerequisites:**
- AWS CLI configured with appropriate permissions
- Agentcore service enabled in your AWS account
- IAM role with Bedrock, DynamoDB, and OpenSearch access

In [ ]:
# Install Agentcore CLI tools
!pip install agentcore-cli

In [ ]:
# Create the agent deployment configuration
agentcore_config = {
    "agent_name": "restaurant-assistant-agent",
    "description": "Multi-tool restaurant assistant with booking management and knowledge base retrieval",
    "runtime": {
        "framework": "strands",
        "entry_point": "agent.py",
        "python_version": "3.11"
    },
    "model_config": {
        "provider": "bedrock",
        "model_id": "anthropic.claude-sonnet-4-20250514-v1:0",
        "region": AWS_REGION,
        "temperature": 0.3,
        "max_tokens": 2048
    },
    "tools": [
        "create_booking",
        "get_booking_details",
        "delete_booking",
        "current_time",
        "retrieve"
    ],
    "environment_variables": {
        "DYNAMODB_TABLE_NAME": DYNAMODB_TABLE_NAME,
        "KNOWLEDGE_BASE_ID": KNOWLEDGE_BASE_ID,
        "AWS_REGION": AWS_REGION
    },
    "iam_role_arn": os.getenv("AGENTCORE_IAM_ROLE_ARN", "arn:aws:iam::role/AgentcoreExecutionRole"),
    "scaling": {
        "min_instances": 1,
        "max_instances": 5
    }
}

# Save configuration
with open("agentcore_config.json", "w") as f:
    json.dump(agentcore_config, f, indent=2)

print("Agentcore deployment configuration saved.")
print(json.dumps(agentcore_config, indent=2))

In [ ]:
# Package the agent for deployment
# This creates the deployment bundle with all tools and dependencies

import shutil

DEPLOY_DIR = "agent_deployment"
os.makedirs(DEPLOY_DIR, exist_ok=True)

# Write the main agent module
agent_code = '''import os
import uuid
import json
import boto3
from datetime import datetime, timezone
from strands import Agent, tool
from strands.models.bedrock import BedrockModel

# Configuration from environment
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
DYNAMODB_TABLE_NAME = os.environ.get("DYNAMODB_TABLE_NAME", "restaurant-bookings")
KNOWLEDGE_BASE_ID = os.environ.get("KNOWLEDGE_BASE_ID")

# AWS clients
dynamodb = boto3.resource("dynamodb", region_name=AWS_REGION)
bookings_table = dynamodb.Table(DYNAMODB_TABLE_NAME)
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)


@tool
def create_booking(restaurant_name: str, customer_name: str, party_size: int,
                   date: str, time: str, special_requests: str = "") -> dict:
    """Create a new restaurant booking."""
    booking_id = str(uuid.uuid4())[:8]
    item = {
        "booking_id": booking_id,
        "restaurant_name": restaurant_name,
        "customer_name": customer_name,
        "party_size": party_size,
        "date": date, "time": time,
        "special_requests": special_requests,
        "status": "confirmed",
        "created_at": datetime.now(timezone.utc).isoformat()
    }
    bookings_table.put_item(Item=item)
    return {"status": "success", "booking_id": booking_id,
            "message": f"Booking confirmed at {restaurant_name} for {customer_name}."}


@tool
def get_booking_details(booking_id: str) -> dict:
    """Retrieve details of an existing restaurant booking."""
    response = bookings_table.get_item(Key={"booking_id": booking_id})
    if "Item" in response:
        return {"status": "found", "booking": response["Item"]}
    return {"status": "not_found", "message": f"No booking found with ID: {booking_id}"}


@tool
def delete_booking(booking_id: str) -> dict:
    """Cancel a restaurant booking."""
    response = bookings_table.get_item(Key={"booking_id": booking_id})
    if "Item" not in response:
        return {"status": "not_found", "message": f"No booking found with ID: {booking_id}"}
    bookings_table.update_item(
        Key={"booking_id": booking_id},
        UpdateExpression="SET #s = :status, cancelled_at = :ts",
        ExpressionAttributeNames={"#s": "status"},
        ExpressionAttributeValues={
            ":status": "cancelled",
            ":ts": datetime.now(timezone.utc).isoformat()
        }
    )
    return {"status": "success", "message": f"Booking {booking_id} cancelled."}


@tool
def current_time() -> dict:
    """Get the current date and time in UTC."""
    now = datetime.now(timezone.utc)
    return {"current_datetime": now.isoformat(), "date": now.strftime("%Y-%m-%d"),
            "time": now.strftime("%H:%M:%S"), "day_of_week": now.strftime("%A")}


@tool
def retrieve(query: str) -> dict:
    """Retrieve information from the restaurant knowledge base."""
    response = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 5}}
    )
    results = [{"content": r["content"]["text"], "score": r.get("score", 0)}
               for r in response.get("retrievalResults", [])]
    return {"query": query, "num_results": len(results), "results": results}


SYSTEM_PROMPT = """
You are a helpful restaurant assistant agent. You help customers with:
- Finding restaurants and their details (menus, hours, locations, policies)
- Making new reservations
- Looking up existing booking details
- Cancelling reservations

Important guidelines:
1. Always use the retrieve tool to search for restaurant information.
2. If a request is impossible or unrealistic, acknowledge it clearly and offer alternatives.
3. Be polite, concise, and accurate.
4. Confirm booking details before creating a reservation.
5. If you don\'t have information, say so honestly.

Format your responses using <answer> tags.
"""


def create_agent():
    model = BedrockModel(
        model_id="anthropic.claude-sonnet-4-20250514-v1:0",
        region_name=AWS_REGION, temperature=0.3, max_tokens=2048
    )
    return Agent(
        model=model, system_prompt=SYSTEM_PROMPT,
        tools=[create_booking, get_booking_details, delete_booking, current_time, retrieve]
    )


# Entry point for Agentcore
agent = create_agent()
'''

with open(os.path.join(DEPLOY_DIR, "agent.py"), "w") as f:
    f.write(agent_code)

# Write requirements.txt
requirements = """strands-agents>=0.1.0
strands-agents-tools>=0.1.0
boto3>=1.34.0
python-dotenv>=1.0.0
"""

with open(os.path.join(DEPLOY_DIR, "requirements.txt"), "w") as f:
    f.write(requirements)

# Copy config
shutil.copy("agentcore_config.json", os.path.join(DEPLOY_DIR, "agentcore_config.json"))

print(f"Deployment package created in '{DEPLOY_DIR}/':")
for fname in os.listdir(DEPLOY_DIR):
    print(f"  - {fname}")

In [ ]:
# Deploy to Agentcore
# Note: Ensure AWS CLI is configured and you have Agentcore permissions

print("Deploying agent to Amazon Bedrock Agentcore...")
print("Run the following command from the 'agent_deployment' directory:")
print()
print("  agentcore deploy --config agentcore_config.json")
print()
print("Or use the AWS CLI:")
print(f"  aws bedrock-agentcore create-agent-runtime \\")
print(f"    --agent-name restaurant-assistant-agent \\")
print(f"    --region {AWS_REGION} \\")
print(f"    --runtime-config file://agentcore_config.json")

## 8. Verify Deployment

After deploying, test the agent through the Agentcore runtime endpoint.

In [ ]:
# Test the deployed agent via Agentcore runtime
# Replace AGENT_RUNTIME_ENDPOINT with your actual endpoint after deployment

import requests

AGENT_RUNTIME_ENDPOINT = os.getenv(
    "AGENTCORE_ENDPOINT",
    "https://<your-agentcore-endpoint>.execute-api.us-east-1.amazonaws.com/prod/invoke"
)

def invoke_deployed_agent(message: str) -> dict:
    """Invoke the deployed agent through Agentcore endpoint."""
    # Using SigV4 signing via boto3 session
    session = boto3.Session(region_name=AWS_REGION)
    credentials = session.get_credentials().get_frozen_credentials()
    
    payload = {
        "input": message,
        "session_id": str(uuid.uuid4())
    }
    
    # For demonstration — in production, use proper SigV4 signing
    response = requests.post(
        AGENT_RUNTIME_ENDPOINT,
        json=payload,
        headers={"Content-Type": "application/json"}
    )
    
    return response.json()


# Test queries against deployed agent
test_queries = [
    "What restaurants do you have in Portland?",
    "Book a table for 2 at the Portland restaurant tomorrow at 6 PM. Name: Sam Lee.",
    "Find me a restaurant on the moon."
]

print("Note: Update AGENTCORE_ENDPOINT before running.")
print("Test queries prepared:")
for i, q in enumerate(test_queries, 1):
    print(f"  {i}. {q}")

---

## Summary

In this notebook, we:
1. Set up AWS resources (DynamoDB, S3, OpenSearch, Bedrock Knowledge Base)
2. Defined five agent tools: `create_booking`, `get_booking_details`, `delete_booking`, `current_time`, `retrieve`
3. Built a Strands SDK agent backed by Amazon Bedrock
4. Tested the agent across multiple scenarios including edge cases
5. Packaged and prepared the agent for deployment on Amazon Bedrock Agentcore

**Next:** Go to **Notebook 2** to add OpenTelemetry tracing, Arize AX observability, LLM-as-a-Judge evaluations, prompt optimization, and production monitoring.